In [ ]:
# purpose: gain experience with models and the sklearn  API

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
# 1. Logistic regression
from sklearn.linear_model import LogisticRegression
# 2. Decision tree
from sklearn.tree import DecisionTreeClassifier
# 3. Random forest
from sklearn.ensemble import RandomForestClassifier
# 4. XGBoost
try:
    from xgboost import XGBClassifier
except ModuleNotFoundError:
    ! pip install xgboost
    from xgboost import XGBClassifier
# 5. Catboost
try:
    from catboost import CatBoostClassifier, Pool
except ModuleNotFoundError:
    ! pip install catboost
    from catboost import CatBoostClassifier, Pool
# 6. LightGBM
try:
    from lightgbm import LGBMClassifier
except ModuleNotFoundError:
    ! pip install lightgbm
    from lightgbm import LGBMClassifier

#### Functions

In [ ]:
class MinMaxScaler:
    def __init__(self):
        pass
    def fit(self, X):
        dict_fit = {}
        for col in X.columns:
            # get min
            flt_min = X[col].min()
            # get max
            flt_max = X[col].max()
            # get range
            flt_range = flt_max - flt_min
            # mak dict
            dict_tmp = {
                'min': flt_min,
                'range': flt_range,
            }
            # assign
            dict_fit[col] = dict_tmp
        # save to object
        self.dict_fit = dict_fit
        # return object
        return self
    def transform(self, X):
        for col, dict_tmp in self.dict_fit.items():
            flt_min = dict_tmp['min']
            flt_range = dict_tmp['range']
            X[f'{col}_scaled'] = (X[col] - flt_min) / flt_range
        # return X 
        return X

#### Constants

In [ ]:
str_dirname_output = './output'

str_target = '60dpd730'

#### Make output dir

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

#### Import data

In [ ]:
str_uri = 'uri_from_aws'
df = pd.read_parquet(str_uri)
df

#### Split into train, test, holdout using data_set column

In [ ]:
df_train = df[df['data_set'] == 'train'].copy()
df_test = df[df['data_set'] == 'test'].copy()
df_holdout = df[df['data_set'] == 'holdout'].copy()

#### List of features

In [ ]:
list_cols_model = [
    # put features here
]

#### Scale features

In [ ]:
cls_scaler = MinMaxScaler()
# fit on train
cls_scaler.fit(df_train[list_cols_model].copy())
# transform
df_train = cls_scaler.transform(df_train)
df_test = cls_scaler.transform(df_test)
df_holdout = cls_scaler.transform(df_holdout)

#### Get scaled features

In [ ]:
list_cols_scaled = [col for col in df_train.columns if 'scaled' in col]

#### 1. Logistic Regression

In [ ]:
list_str_penalty = [
    'l1',
    'l2',
    'elasticnet',
    None,
]
list_dict_row = []
for str_penalty in tqdm(list_str_penalty):
    # logic for conpatibility
    if str_penalty == 'l1':
        str_solver = 'liblinear'
        l1_ratio = None
    elif str_penalty == 'elasticnet':
        str_solver = 'saga'
        l1_ratio = 0.5
    else:
        str_solver = 'lbfgs'
        l1_ratio = None
    
    # initialize model
    cls_model_inference = LogisticRegression(
        random_state=42,
        fit_intercept=True,
        penalty=str_penalty,
        solver=str_solver,
        l1_ratio=l1_ratio,
    )

    # fit on train
    cls_model_inference.fit(
        df_train[list_cols_scaled].copy(),
        df_train[str_target],
    )

    # predict on train (in-sample)
    yhat_train = cls_model_inference.predict_proba(df_train[list_cols_scaled])[:, 1]
    # predict on test (out of sample)
    yhat_test = cls_model_inference.predict_proba(df_test[list_cols_scaled])[:, 1]
    # predict on holdout (out of sample)
    yhat_holdout = cls_model_inference.predict_proba(df_holdout[list_cols_scaled])[:, 1]

    # evaluate on train
    flt_score_train = roc_auc_score(y_true=df_train[str_target], y_score=yhat_train)
    # evaluate on test
    flt_score_test = roc_auc_score(y_true=df_test[str_target], y_score=yhat_test)
    # evaluate on holdout
    flt_score_holdout = roc_auc_score(y_true=df_holdout[str_target], y_score=yhat_holdout)

    # put into dictionary
    dict_row = {
        'penalty': str_penalty,
        'train': flt_score_train,
        'test': flt_score_test,
        'holdout': flt_score_holdout,
    }
    # append
    list_dict_row.append(dict_row)

# make df
df_tmp = pd.DataFrame(list_dict_row)
df_tmp['penalty'] = df_tmp['penalty'].astype(str)
# show
df_tmp

In [ ]:
str_x = 'penalty'
x = df_tmp[str_x]
list_cols = [col for col in df_tmp.columns if col != str_x]
fig, ax = plt.subplots(figsize=(9,5))
ax.set_title('Score by Data Set by Hyperparameter')
ax.set_xlabel('Hyperparameter')
ax.set_ylabel('Score')
for col in tqdm(list_cols):
    ax.plot(x, df_tmp[col], label=col)
ax.legend()
plt.show()

#### 2. Decision tree

In [ ]:
list_dict_row = []
for int_max_depth in tqdm(range(1, 7)):
    # initialize model
    cls_model_inference = DecisionTreeClassifier(
        random_state=42,
        max_depth=int_max_depth,
    )

    # fit on train
    cls_model_inference.fit(
        df_train[list_cols_model].copy(),
        df_train[str_target],
    )

    # predict on train (in-sample)
    yhat_train = cls_model_inference.predict_proba(df_train[list_cols_model])[:, 1]
    # predict on test (out of sample)
    yhat_test = cls_model_inference.predict_proba(df_test[list_cols_model])[:, 1]
    # predict on holdout (out of sample)
    yhat_holdout = cls_model_inference.predict_proba(df_holdout[list_cols_model])[:, 1]

    # evaluate on train
    flt_score_train = roc_auc_score(y_true=df_train[str_target], y_score=yhat_train)
    # evaluate on test
    flt_score_test = roc_auc_score(y_true=df_test[str_target], y_score=yhat_test)
    # evaluate on holdout
    flt_score_holdout = roc_auc_score(y_true=df_holdout[str_target], y_score=yhat_holdout)

    # put into dictionary
    dict_row = {
        'max_depth': int_max_depth,
        'train': flt_score_train,
        'test': flt_score_test,
        'holdout': flt_score_holdout,
    }
    # append
    list_dict_row.append(dict_row)
    
# make df
df_tmp = pd.DataFrame(list_dict_row)
df_tmp['max_depth'] = df_tmp['max_depth'].astype(str)
# show
df_tmp

In [ ]:
str_x = 'max_depth'
x = df_tmp[str_x]
list_cols = [col for col in df_tmp.columns if col != str_x]
fig, ax = plt.subplots(figsize=(9,5))
ax.set_title('Score by Data Set by Hyperparameter')
ax.set_xlabel('Hyperparameter')
ax.set_ylabel('Score')
for col in tqdm(list_cols):
    ax.plot(x, df_tmp[col], label=col)
ax.legend()
plt.show()

#### 3. Random forest

In [ ]:
list_dict_row = []
for int_max_depth in tqdm(range(1, 7)):
    # initialize model
    cls_model_inference = RandomForestClassifier(
        random_state=42,
        max_depth=int_max_depth,
    )

    # fit on train
    cls_model_inference.fit(
        df_train[list_cols_model].copy(),
        df_train[str_target],
    )

    # predict on train (in-sample)
    yhat_train = cls_model_inference.predict_proba(df_train[list_cols_model])[:, 1]
    # predict on test (out of sample)
    yhat_test = cls_model_inference.predict_proba(df_test[list_cols_model])[:, 1]
    # predict on holdout (out of sample)
    yhat_holdout = cls_model_inference.predict_proba(df_holdout[list_cols_model])[:, 1]

    # evaluate on train
    flt_score_train = roc_auc_score(y_true=df_train[str_target], y_score=yhat_train)
    # evaluate on test
    flt_score_test = roc_auc_score(y_true=df_test[str_target], y_score=yhat_test)
    # evaluate on holdout
    flt_score_holdout = roc_auc_score(y_true=df_holdout[str_target], y_score=yhat_holdout)

    # put into dictionary
    dict_row = {
        'max_depth': int_max_depth,
        'train': flt_score_train,
        'test': flt_score_test,
        'holdout': flt_score_holdout,
    }
    # append
    list_dict_row.append(dict_row)
    
# make df
df_tmp = pd.DataFrame(list_dict_row)
df_tmp['max_depth'] = df_tmp['max_depth'].astype(str)
# show
df_tmp

In [ ]:
str_x = 'max_depth'
x = df_tmp[str_x]
list_cols = [col for col in df_tmp.columns if col != str_x]
fig, ax = plt.subplots(figsize=(9,5))
ax.set_title('Score by Data Set by Hyperparameter')
ax.set_xlabel('Hyperparameter')
ax.set_ylabel('Score')
for col in tqdm(list_cols):
    ax.plot(x, df_tmp[col], label=col)
ax.legend()
plt.show()

#### 4. XGBoost

In [ ]:
# initialize

# fit

# predict

# evaluate

#### 5. Catboost

In [ ]:
# initialize

# fit

# predict

# evaluate

#### 6. LightGBM

In [ ]:
# initialize

# fit

# predict

# evaluate